# kronos-semi · Notebook 05: Axisymmetric MOSCAP C-V (LF vs HF)

Reproduces the low-frequency (LF) and high-frequency (HF) C-V curves of the MOS capacitor in Hu, *Modern Semiconductor Devices for Integrated Circuits*, Fig. 5-18, on an **axisymmetric (cylindrical) 2D** mesh.

## Geometry

We solve in the meridian half-plane $(r, z)$ with rotational symmetry about the $z$-axis:

- $r \in [0, R = 200~\mu\mathrm{m}]$, $z \in [-T_\mathrm{Si}, T_\mathrm{ox}] = [-5~\mu\mathrm{m}, 10~\mathrm{nm}]$.
- Gate Dirichlet on $z = T_\mathrm{ox}$, $r \in [0, R_g = 50~\mu\mathrm{m}]$.
- Bulk ohmic on $z = -T_\mathrm{Si}$.
- Symmetry axis $r = 0$: natural no-flux (no Dirichlet, enforced by schema).
- Outer wall $r = R$: homogeneous Neumann; $R \gg 5\,W_{d\mathrm{max}}$ so radial truncation is negligible.

## Weak forms

Cylindrical-symmetry forms multiply every volume integrand by $r$:
$$\int A\,\nabla u \cdot \nabla v\;r\,dr\,dz \;=\; \int f\,v\;r\,dr\,dz.$$
Implemented in [`semi/physics/axisymmetric.py`](../semi/physics/axisymmetric.py).

## C-V extraction

Two curves are extracted, both from the *same* DC operating points:

- **LF** (quasi-static): $C_\mathrm{LF} = -dQ_s/dV_g$ via centered FD on the $r$-weighted total semiconductor charge (Hu Eq. 5.6.1).
- **HF** (depletion clamp): from the FEM-extracted surface potential, with $W_\mathrm{dep}$ clamped at $W_{d\mathrm{max}}$ once $\varphi_s \ge 2\varphi_B$. This is the chosen HF method for this benchmark; a fully principled small-signal solve with frozen minority carriers is a future extension.

Both routines live in [`semi/cv.py`](../semi/cv.py).

## Status

**Draft scaffold.** This notebook is committed without executed FEM outputs because the local environment lacks `dolfinx`, `gmsh`, and `pyvista`. The cells below run the **analytical reference** path (pure NumPy + SciPy) that the FEM result is compared against; reproducing the full FEM curves requires the kronos-semi Docker image or Colab with `dolfinx`.

The intended figure list (per the issue acceptance criteria) is:

1. Mesh (meridian half-plane, with gate edge refinement).
2. Equilibrium $\psi(r, z)$ contour.
3. Strong-inversion $\psi(r, z)$ contour at $V_g = +2$ V.
4. Charge density $\rho(r, z)$ at the threshold bias.
5. Surface potential $\varphi_s(r)$ vs. $V_g$ family.
6. LF and HF C-V on the same axes (this is the headline figure).
7. Radial-truncation convergence: $C(V_g{=}0)$ vs. $R/W_{d\mathrm{max}}$.

Items 1, 6 (FEM curve) and 7 require dolfinx; we render item 6's analytical baseline below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from semi.cv import (
    analytical_moscap_params,
    hf_cv_depletion_approximation,
    lf_cv_quasistatic,
)

params = analytical_moscap_params(
    body_dopant="p",
    N_body_cm3=5.0e16,
    T_ox_m=10.0e-9,
    phi_ms=-0.95,
)
print(f"V_fb     = {params.V_fb:+.3f} V")
print(f"V_t      = {params.V_t:+.3f} V")
print(f"|phi_B|  = {params.phi_B:+.3f} V")
print(f"W_dmax   = {params.W_dmax * 1e9:6.1f} nm")
print(f"C_ox     = {params.C_ox_per_area * 1e3:6.3f} mF/m^2")
print(f"C_min/Cox= {params.C_min_per_area / params.C_ox_per_area:6.3f}")

In [ ]:
Vg = np.linspace(-2.0, 2.0, 81)
C_LF = lf_cv_quasistatic(Vg, params, N_body_cm3=5.0e16)
C_HF = hf_cv_depletion_approximation(Vg, params, N_body_cm3=5.0e16)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(Vg, C_LF, label="low frequency (quasi-static)")
ax.plot(Vg, C_HF, label="high frequency (depletion clamp)")
ax.axvline(params.V_fb, color="0.6", ls=":", lw=1, label=f"V_fb = {params.V_fb:+.2f} V")
ax.axvline(params.V_t,  color="0.4", ls="--", lw=1, label=f"V_t = {params.V_t:+.2f} V")
ax.set_xlabel("V_g  (V)")
ax.set_ylabel("C / C_ox")
ax.set_ylim(0, 1.05)
ax.set_title("Analytical MOSCAP C-V (Hu Fig. 5-18 reference)")
ax.grid(True, ls=":", alpha=0.5)
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig("../results/moscap_axisym_2d/cv_analytical.png", dpi=140) if False else None

## Reproducing the full FEM curves

From the kronos-semi Docker image:

```bash
# 1. Build the mesh
gmsh -2 -format msh22 \
  -o benchmarks/moscap_axisym_2d/moscap_axisym.msh \
  benchmarks/moscap_axisym_2d/moscap_axisym.geo

# 2. Run the benchmark
python scripts/run_benchmark.py benchmarks/moscap_axisym_2d/moscap_axisym.json

# 3. Re-render this notebook
jupyter nbconvert --to notebook --execute notebooks/05_moscap_axisym_cv.ipynb
```

The C-V regression check is in `tests/test_moscap_axisym_cv.py` (gated by `pytest.importorskip('dolfinx')`).